In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


from collections import Counter
from getpass import getpass
import os
from pathlib import Path
import random
import shutil
import zipfile

from IPython.display import FileLink
import numpy as np
pandas as pd
from roboflow import Roboflow
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_B2_Weights, efficientnet_b2
from tqdm.auto import tqdm
from ultralytics import YOLO

/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/desktop.ini
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (70).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (55).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (26).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (23).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (53).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (51).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (13).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (24).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (22).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (16).jpg
/kaggle/input/datasets/lucky3442/pests-de/Orig

In [ ]:


for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/desktop.ini
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (70).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (55).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (26).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (23).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (53).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (51).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (13).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (24).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (22).jpg
/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest/TU/TU (16).jpg
/kaggle/input/datasets/lucky3442/pests-de/Orig

In [ ]:


for p in Path("/kaggle/input").rglob("*.7z"):
    print(p)
    print(f"Size: {p.stat().st_size / (1024*1024):.2f} MB")

In [5]:
DATASET = "/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest"

In [ ]:


DATASET = Path(
    "/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest"
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

classes = sorted([
    d for d in DATASET.iterdir()
    if d.is_dir()
])


print("Dataset:", DATASET)
print("Classes:", len(classes))
print()

total = 0

for cls in classes:
    images = [
        p for p in cls.iterdir()
        if p.is_file() and p.suffix in IMAGE_EXTS
    ]

    print(f"{cls.name:5} : {len(images)} images")
    total += len(images)

print("\nTotal images:", total)

Dataset: /kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest
Classes: 8

BA    : 60 images
HA    : 108 images
MP    : 130 images
SE    : 75 images
SL    : 96 images
TP    : 24 images
TU    : 74 images
ZC    : 42 images

Total images: 609


In [7]:
CLASS_NAMES = {
    "TU": "Two-spotted Spider Mite",
    "BA": "Beet Armyworm",
    "ZC": "Melon Fruit Fly",
    "SL": "Serpentine Leaf Miner",
    "MP": "Green Peach Aphid",
    "TP": "Melon Thrips",
    "SE": "Tobacco Cutworm",
    "HA": "Fruit Borer",
}

for folder, name in CLASS_NAMES.items():
    print(f"{folder} -> {name}")

TU -> Two-spotted Spider Mite
BA -> Beet Armyworm
ZC -> Melon Fruit Fly
SL -> Serpentine Leaf Miner
MP -> Green Peach Aphid
TP -> Melon Thrips
SE -> Tobacco Cutworm
HA -> Fruit Borer


In [ ]:


DATASET = Path(
    "/kaggle/input/datasets/lucky3442/pests-de/Original image of tomato pest"
)

OUTPUT = Path("/kaggle/working/tomato_pest_classifier")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

# Collect all images with their folder/class
data = []

for class_dir in sorted(DATASET.iterdir()):
    if not class_dir.is_dir():
        continue

    for img in class_dir.iterdir():
        if img.is_file() and img.suffix in IMAGE_EXTS:
            data.append((img, class_dir.name))

print("Total:", len(data))

Total: 609


In [9]:
paths = [x[0] for x in data]
labels = [x[1] for x in data]

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train:", len(train_paths))
print("Val  :", len(val_paths))
print("Test :", len(test_paths))

Train: 426
Val  : 91
Test : 92


In [10]:
from collections import Counter

print("\nTRAIN")
print(Counter(train_labels))

print("\nVAL")
print(Counter(val_labels))

print("\nTEST")
print(Counter(test_labels))



TRAIN
Counter({'MP': 91, 'HA': 76, 'SL': 67, 'SE': 52, 'TU': 52, 'BA': 42, 'ZC': 29, 'TP': 17})

VAL
Counter({'MP': 19, 'HA': 16, 'SL': 14, 'SE': 11, 'TU': 11, 'BA': 9, 'ZC': 7, 'TP': 4})

TEST
Counter({'MP': 20, 'HA': 16, 'SL': 15, 'SE': 12, 'TU': 11, 'BA': 9, 'ZC': 6, 'TP': 3})


In [ ]:


OUTPUT = Path("/kaggle/working/tomato_pest_classifier")

# Remove previous split if it exists
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

splits = {
    "train": (train_paths, train_labels),
    "val":   (val_paths, val_labels),
    "test":  (test_paths, test_labels)
}

for split, (paths, labels) in splits.items():

    for img_path, label in zip(paths, labels):

        dest_dir = OUTPUT / split / label
        dest_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            img_path,
            dest_dir / img_path.name
        )

print("Dataset created at:", OUTPUT)

Dataset created at: /kaggle/working/tomato_pest_classifier


In [ ]:


for split in ["train", "val", "test"]:

    counts = Counter()

    for class_dir in (OUTPUT / split).iterdir():
        if class_dir.is_dir():
            counts[class_dir.name] = len([
                p for p in class_dir.iterdir()
                if p.is_file()
            ])

    print(f"\n{split.upper()}")
    print(counts)
    print("Total:", sum(counts.values()))


TRAIN
Counter({'MP': 91, 'HA': 76, 'SL': 67, 'SE': 52, 'TU': 52, 'BA': 42, 'ZC': 29, 'TP': 17})
Total: 426

VAL
Counter({'MP': 19, 'HA': 16, 'SL': 14, 'SE': 11, 'TU': 11, 'BA': 9, 'ZC': 7, 'TP': 4})
Total: 91

TEST
Counter({'MP': 20, 'HA': 16, 'SL': 15, 'SE': 12, 'TU': 11, 'BA': 9, 'ZC': 6, 'TP': 3})
Total: 92


In [ ]:

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
CUDA: True
Device: cuda


In [ ]:


DATA_ROOT = "/kaggle/working/tomato_pest_classifier"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(
        224,
        scale=(0.75, 1.0)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    f"{DATA_ROOT}/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    f"{DATA_ROOT}/val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    f"{DATA_ROOT}/test",
    transform=eval_transform
)

print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Classes: ['BA', 'HA', 'MP', 'SE', 'SL', 'TP', 'TU', 'ZC']
Train: 426
Val: 91
Test: 92


In [ ]:


class_counts = Counter(train_dataset.targets)

class_weights = {
    cls: 1.0 / count
    for cls, count in class_counts.items()
}

sample_weights = [
    class_weights[label]
    for label in train_dataset.targets
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:

# FINAL TOMATO PEST CLASSIFIER
# EfficientNet-B2 + balanced sampling + augmentation






SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



# 3. PATHS


DATA_ROOT = Path(
    "/kaggle/working/tomato_pest_classifier"
)

MODEL_PATH = (
    "/kaggle/working/"
    "best_tomato_pest_classifier.pt"
)



# IMAGE TRANSFORMS


IMG_SIZE = 260

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]


# Training augmentation
#
# Important:
# These transformations are applied ONLY to training images.
#
train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.70, 1.0),
        ratio=(0.80, 1.25)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomVerticalFlip(
        p=0.15
    ),

    transforms.RandomRotation(
        degrees=25
    ),

    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.20,
        hue=0.05
    ),

    transforms.RandomApply(
        [
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 1.5)
            )
        ],
        p=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    ),

    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.10),
        ratio=(0.3, 3.3)
    )
])


# Validation/test:
# NO augmentation.
eval_transform = transforms.Compose([

    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=mean,
        std=std
    )
])


# datset
train_dataset = datasets.ImageFolder(
    DATA_ROOT / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    DATA_ROOT / "val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    DATA_ROOT / "test",
    transform=eval_transform
)




print("Classes:")
for i, name in enumerate(train_dataset.classes):
    print(i, ":", name)

print("\nTrain:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))



# 6. VERIFY CLASS ORDER


assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

NUM_CLASSES = len(train_dataset.classes)

print("\nClass mapping:")
print(train_dataset.class_to_idx)



# 7. BALANCED SAMPLER


train_targets = train_dataset.targets

class_counts = Counter(train_targets)

print("\n TRAIN CLASS COUNTS ")

for class_idx in range(NUM_CLASSES):

    print(
        train_dataset.classes[class_idx],
        ":",
        class_counts[class_idx]
    )


# Inverse-frequency weighting
class_weights = {
    cls: 1.0 / count
    for cls, count in class_counts.items()
}


sample_weights = [
    class_weights[label]
    for label in train_targets
]


sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)


#  DATALOADERS

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)



#  PRETRAINED EFFICIENTNET-B2


weights = EfficientNet_B2_Weights.DEFAULT

model = efficientnet_b2(
    weights=weights
)

# Replace classifier
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(device)


print("\nModel:")
print("EfficientNet-B2")
print("Parameters:",
      sum(p.numel() for p in model.parameters()))



# LOSS

#
# We already use balanced sampling.
# Therefore DON'T additionally apply inverse class
# weights to CrossEntropyLoss.
#
# Label smoothing improves generalization on small datasets.

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.10
)



# OPTIMIZER


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)



# COSINE LR SCHEDULER


EPOCHS = 40

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)




use_amp = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)


# TRAINING FUNCTION


def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(
        train_loader,
        desc="Training",
        leave=False
    )

    for images, labels in progress:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        progress.set_postfix(
            loss=loss.item()
        )

    return (
        running_loss / total,
        correct / total
    )


#  VALIDATION


@torch.no_grad()
def validate():

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in val_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type=device.type,
            enabled=use_amp
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        all_preds.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    val_loss = (
        running_loss /
        len(val_dataset)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return (
        val_loss,
        val_accuracy,
        val_macro_f1
    )


#  FINAL TRAINING


best_macro_f1 = -1

history = []



for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc, val_macro_f1 = validate()

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "train_accuracy": train_acc,

        "val_loss": val_loss,

        "val_accuracy": val_acc,

        "val_macro_f1": val_macro_f1,

        "lr": current_lr
    })


    print(
        f"\nEpoch {epoch+1:02d}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.4f}"
        f" | Val Macro-F1: {val_macro_f1:.4f}"
        f" | LR: {current_lr:.7f}"
    )


    # Save BEST model
    if val_macro_f1 > best_macro_f1:

        best_macro_f1 = val_macro_f1

        torch.save({

            "model_state_dict":
                model.state_dict(),

            "class_to_idx":
                train_dataset.class_to_idx,

            "classes":
                train_dataset.classes,

            "best_val_macro_f1":
                best_macro_f1,

            "epoch":
                epoch + 1

        }, MODEL_PATH)

        print(
            ">>> BEST MODEL SAVED"
        )




print(
    "Best validation Macro-F1:",
    best_macro_f1
)

print(
    "Saved:",
    MODEL_PATH
)

Device: cuda
GPU: Tesla T4

===== DATASET =====
Classes:
0 : BA
1 : HA
2 : MP
3 : SE
4 : SL
5 : TP
6 : TU
7 : ZC

Train: 426
Val  : 91
Test : 92

Class mapping:
{'BA': 0, 'HA': 1, 'MP': 2, 'SE': 3, 'SL': 4, 'TP': 5, 'TU': 6, 'ZC': 7}

===== TRAIN CLASS COUNTS =====
BA : 42
HA : 76
MP : 91
SE : 52
SL : 67
TP : 17
TU : 52
ZC : 29
Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 152MB/s] 



Model:
EfficientNet-B2
Parameters: 7712266

STARTING FINAL TRAINING


Training:   0%|          | 0/14 [00:00<?, ?it/s]


Epoch 01/40 | Train Loss: 1.7949 | Train Acc: 0.4554 | Val Loss: 1.4905 | Val Acc: 0.6593 | Val Macro-F1: 0.6731 | LR: 0.0002995
>>> BEST MODEL SAVED


...


Epoch 38/40 | Train Loss: 0.4902 | Train Acc: 0.9977 | Val Loss: 0.7221 | Val Acc: 0.9011 | Val Macro-F1: 0.8831 | LR: 0.0000028


Training:   0%|          | 0/14 [00:00<?, ?it/s]


Epoch 39/40 | Train Loss: 0.4887 | Train Acc: 1.0000 | Val Loss: 0.7222 | Val Acc: 0.9011 | Val Macro-F1: 0.8831 | LR: 0.0000015


Training:   0%|          | 0/14 [00:00<?, ?it/s]


Epoch 40/40 | Train Loss: 0.4928 | Train Acc: 0.9977 | Val Loss: 0.7342 | Val Acc: 0.9011 | Val Macro-F1: 0.8831 | LR: 0.0000010

TRAINING FINISHED
Best validation Macro-F1: 0.9098183838752893
Saved: /kaggle/working/best_tomato_pest_classifier.pt


In [17]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "/kaggle/working/tomato_pest_training_history.csv",
    index=False
)

history_df.tail()

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_f1,lr
35,36,0.511509,0.988263,0.747517,0.890110,0.875683,0.000008
36,37,0.502954,0.990610,0.726157,0.890110,0.875683,0.000005
37,38,0.490154,0.997653,0.722068,0.901099,0.883113,0.000003
38,39,0.488716,1.000000,0.722199,0.901099,0.883113,0.000001
39,40,0.492810,0.997653,0.734192,0.901099,0.883113,0.000001


In [ ]:
#FINAL PERFORMANCE + SAVE MODEL AS KAGGLE OUTPUT








MODEL_PATH = "/kaggle/working/best_tomato_pest_classifier.pt"

ZIP_PATH = "/kaggle/working/tomato_pest_classifier_final.zip"

print("Model exists:", os.path.exists(MODEL_PATH))

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}"
    )




checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

class_names = checkpoint["classes"]


print("Best epoch:", checkpoint["epoch"])
print(
    "Best validation Macro-F1:",
    f"{checkpoint['best_val_macro_f1']:.4f}"
)




all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in tqdm(
        test_loader,
        desc="Evaluating TEST set"
    ):

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = outputs.argmax(
            dim=1
        )

        all_preds.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )



accuracy = accuracy_score(
    all_labels,
    all_preds
)

precision_macro, recall_macro, f1_macro, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )
)

precision_weighted, recall_weighted, f1_weighted, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )
)



print("FINAL TOMATO PEST CLASSIFIER PERFORMANCE")


print(f"Accuracy            : {accuracy:.4f}")
print(f"Macro Precision     : {precision_macro:.4f}")
print(f"Macro Recall        : {recall_macro:.4f}")
print(f"Macro F1            : {f1_macro:.4f}")

print("\nWeighted metrics:")
print(f"Weighted Precision  : {precision_weighted:.4f}")
print(f"Weighted Recall     : {recall_weighted:.4f}")
print(f"Weighted F1         : {f1_weighted:.4f}")



#  Per-class performance


print("\nPER-CLASS PERFORMANCE\n")

report = classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)


# Confusion matrix


cm = confusion_matrix(
    all_labels,
    all_preds
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\nCONFUSION MATRIX ")
print(cm_df)




metrics = {
    "accuracy": accuracy,
    "macro_precision": precision_macro,
    "macro_recall": recall_macro,
    "macro_f1": f1_macro,
    "weighted_precision": precision_weighted,
    "weighted_recall": recall_weighted,
    "weighted_f1": f1_weighted,
    "best_epoch": checkpoint["epoch"],
    "best_validation_macro_f1": checkpoint["best_val_macro_f1"]
}

metrics_df = pd.DataFrame(
    [metrics]
)

metrics_path = (
    "/kaggle/working/"
    "tomato_pest_final_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)



cm_path = (
    "/kaggle/working/"
    "tomato_pest_confusion_matrix.csv"
)

cm_df.to_csv(cm_path)




zip_files = [
    MODEL_PATH,
    metrics_path,
    cm_path
]

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for file in zip_files:

        z.write(
            file,
            arcname=os.path.basename(file)
        )



print("KAGGLE OUTPUT")


for file in zip_files + [ZIP_PATH]:

    size_mb = os.path.getsize(file) / (1024 * 1024)

    print(
        f"{os.path.basename(file):45} "
        f"{size_mb:.2f} MB"
    )

print("\nZIP created:")
print(ZIP_PATH)

print("=" * 65)

Model exists: True

===== BEST CHECKPOINT =====
Best epoch: 32
Best validation Macro-F1: 0.9098


Evaluating TEST set:   0%|          | 0/3 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fbeb591d300><function _MultiProcessingDataLoaderIter.__del__ at 0x7fbeb591d300>
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        if w.is_alive():
if w.is_alive(): 
           ^ ^ ^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python



FINAL TOMATO PEST CLASSIFIER PERFORMANCE
Accuracy            : 0.8913
Macro Precision     : 0.9075
Macro Recall        : 0.8747
Macro F1            : 0.8854

Weighted metrics:
Weighted Precision  : 0.8951
Weighted Recall     : 0.8913
Weighted F1         : 0.8900

===== PER-CLASS PERFORMANCE =====

              precision    recall  f1-score   support

          BA     0.8889    0.8889    0.8889         9
          HA     0.8421    1.0000    0.9143        16
          MP     0.9444    0.8500    0.8947        20
          SE     0.8182    0.7500    0.7826        12
          SL     0.9333    0.9333    0.9333        15
          TP     1.0000    0.6667    0.8000         3
          TU     0.8333    0.9091    0.8696        11
          ZC     1.0000    1.0000    1.0000         6

    accuracy                         0.8913        92
   macro avg     0.9075    0.8747    0.8854        92
weighted avg     0.8951    0.8913    0.8900        92


===== CONFUSION MATRIX =====
    BA  HA  MP  SE